In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')

In [3]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce').fillna(0)
data['ChurnFlag'] = (data['Churn'] == 'Yes').astype(int)

In [4]:
df = data.drop(columns=['customerID', 'Churn'])
print(df.shape)

(7043, 20)


In [5]:
X = df.drop(columns='ChurnFlag')
y = df['ChurnFlag']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

Stratify mostly used in Classification problem , to balance out the train-test split.

In [7]:
print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(5634, 19) (1409, 19)
0.265 0.265


In [8]:
#Get mutual information scores for all features
from sklearn.feature_selection import mutual_info_classif

Xt = X_train.copy()
cat_cols = Xt.select_dtypes('object').columns
for c in cat_cols:
    Xt[c] = Xt[c].astype('category').cat.codes   # temporary codes, for ranking only

discrete = [c in cat_cols or c == 'SeniorCitizen' for c in Xt.columns]
mi = mutual_info_classif(Xt, y_train, discrete_features=discrete, random_state=42)
pd.Series(mi, index=Xt.columns).sort_values(ascending=False).round(3)

/tmp/ipykernel_22035/4011603173.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = Xt.select_dtypes('object').columns


Contract            0.099
tenure              0.079
OnlineSecurity      0.066
TechSupport         0.064
InternetService     0.057
MonthlyCharges      0.048
OnlineBackup        0.047
PaymentMethod       0.047
DeviceProtection    0.044
TotalCharges        0.040
StreamingMovies     0.032
StreamingTV         0.032
PaperlessBilling    0.020
Dependents          0.015
Partner             0.011
SeniorCitizen       0.010
MultipleLines       0.001
PhoneService        0.000
gender              0.000
dtype: float64

Since Gender , phoneService and Multiple Lines have less MI scores , we drop these column

In [9]:
drop_cols = ['gender', 'PhoneService', 'MultipleLines']
X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)
print(X_train.shape) 

(5634, 16)


In [10]:
def add_features(d):
    d = d.copy()
    addons = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies']

    # Churn drops sharply with tenure, so give the model coarse lifecycle stages
    d['tenure_group'] = pd.cut(d['tenure'], bins=[-1, 12, 24, 48, 72],
                               labels=['0-12', '13-24', '25-48', '49-72']).astype(str)

    # Total add-ons subscribed, plus the two strongest protective ones
    d['num_addons'] = (d[addons] == 'Yes').sum(axis=1)
    d['has_security_support'] = ((d['OnlineSecurity'] == 'Yes') & (d['TechSupport'] == 'Yes')).astype(int)

    # The highest-risk combo from EDA
    d['mtm_echeck'] = ((d['Contract'] == 'Month-to-month') &
                       (d['PaymentMethod'] == 'Electronic check')).astype(int)

    # Actual average spend vs. current price (price-hike signal); tenure 0 falls back to MonthlyCharges
    avg_spend = (d['TotalCharges'] / d['tenure'].replace(0, np.nan)).fillna(d['MonthlyCharges'])
    d['charge_change'] = d['MonthlyCharges'] - avg_spend
    return d

In [11]:
X_train = add_features(X_train)
X_test = add_features(X_test)

In [12]:
print(X_train.shape)

(5634, 21)


In [13]:
X_train[['tenure_group', 'num_addons', 'has_security_support', 'mtm_echeck', 'charge_change']].head()

,tenure_group,num_addons,has_security_support,mtm_echeck,charge_change
3738,25-48,3,0,1,0.581429
3151,13-24,1,0,0,-1.670000
4860,13-24,3,1,0,-4.861538
3867,25-48,4,0,0,0.203846
3810,0-12,0,0,1,0.000000


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X_train.select_dtypes('object').columns.tolist()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charge_change', 'num_addons']
bin_cols = ['SeniorCitizen', 'has_security_support', 'mtm_echeck'] 

/tmp/ipykernel_22035/3573002347.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes('object').columns.tolist()


In [15]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols),
    ('bin', 'passthrough', bin_cols),
], sparse_threshold=0)


In [16]:
X_train_enc = preprocessor.fit_transform(X_train)   # fit on train only
X_test_enc = preprocessor.transform(X_test) 

In [17]:
feature_names = preprocessor.get_feature_names_out()
print(X_train_enc.shape, X_test_enc.shape)
print(len(feature_names), 'features')

(5634, 43) (1409, 43)
43 features


In [18]:
print(np.isnan(X_train_enc).sum())              # should be 0
print(X_train_enc[:, :3].mean(axis=0).round(2))  # eyeball a few columns

0
[0.48 0.3  0.34]


In [19]:
import joblib, os
import pickle
os.makedirs('../data/processed', exist_ok=True)

X_train.to_pickle('../data/processed/X_train.pkl')
X_test.to_pickle('../data/processed/X_test.pkl')
y_train.to_pickle('../data/processed/y_train.pkl')
y_test.to_pickle('../data/processed/y_test.pkl')
joblib.dump(preprocessor, '../models/preprocessor.joblib')   # create a models/ folder first

['../models/preprocessor.joblib']